# Scanning Session Ingestion

## Setup

### Connect to the database

If you are don't have your login information, contact the administrator.

Using local config file (see [01_pipeline_activation](./01_pipeline_activation.ipynb)):

https://github.com/datajoint/workflow-calcium-imaging/blob/main/notebooks/03-process.ipynb
https://github.com/datajoint/workflow-calcium-imaging/blob/main/notebooks/04-automate-optional.ipynb

Check the Moser lab descriptions of useage as well.
https://github.com/kavli-ntnu/dj-docs/blob/master/notebooks/2022-11%20Imaging%20workshop/Working_with_Imaging_pipeline.ipynb

Check the youtube: https://www.youtube.com/watch?v=gFLn0GB1L30

In [ ]:
import os
# change to the upper level folder to detect dj_local_conf.json
from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir('..')
import datajoint as dj; dj.conn()

from adamacs.pipeline import subject, session, surgery, scan, event, trial, imaging
from adamacs import utility
from adamacs.ingest import behavior as ibe
import numpy as np
import scanreader

Manual entry:

In [ ]:
import os
# Manual Entry
import datajoint as dj; import getpass
dj.config['database.host'] = os.environ.get('DJ_HOST', 'localhost')        # Put the server name between these apostrophe
dj.config['database.user'] = os.environ.get('DJ_USER', 'dj_user')             # Put your user name between these apostrophe
dj.config['database.password'] = getpass.getpass()  # Put your password in the prompt
dj.conn()

from adamacs.pipeline import subject, session, surgery, scan, event, trial, imaging
from adamacs import utility
from adamacs.ingest import behavior as ibe
import numpy as np

## populate scaninfo

In [ ]:
scan.ScanInfo.describe()

In [ ]:
scan.ScanInfo.heading

In [ ]:
dj.Diagram(subject.Subject) + dj.Diagram(session.Session) + dj.Diagram(scan) + dj.Diagram(imaging.Processing) + dj.Diagram(imaging)

In [ ]:
dj.Diagram(imaging)

In [ ]:
dj.Diagram(scan)

In [ ]:
#scan.ScanInfo.delete()
#scan.Scan.delete()
#session.Session.delete()

In [ ]:
scan.Scan()

In [ ]:
scan.Scan() * scan.ScanPath() * session.Session() * session.SessionNote()

In [ ]:
scan.ScanLocation()

## populate scan!

In [ ]:
 populate_settings = {'display_progress': True, 'suppress_errors': True}

In [ ]:
scan.ScanInfo.populate(**populate_settings)
scan.ScanInfo()

In [ ]:
scan.Scan() * scan.ScanLocation()

In [ ]:
scan.ScanInfo()

Example to update an entry post-hoc. TODO: imnplement in elements "Scan" function based on userfunction stringa

In [ ]:
scan.Scan.update1({'session_id': 'sess9FGLEFJ3', 'scan_id': 'scan9FGLEFJ3', 'scan_notes': "Awesome scan"})

In [ ]:
scan.ScanLocation.update1({'session_id': 'sess9FGLEFJ3', 'scan_id': 'scan9FGLEFJ3', 'anatomical_location': "dCA1"})

In [ ]:
session.SessionNote.insert1({'session_id': 'sess9FGLEFJ3', 'session_note': "Awesome session"})

In [ ]:
session.ProjectSession()

In [ ]:
scan.Scan() 

In [ ]:
scan.ScanLocation()

### Creating a Parameter Set

What exactly happens during processing dependso on the parameter set. This is an example of a parameter set and its insert:

In [ ]:
dj.config['custom'].get('suite2p_fast_tmp')[0]

In [ ]:
# Insert the param_set - Mini2p

# TODO: Parameter set needs to be updated with ScanInfo settings

params_suite2p = {'look_one_level_down': False,
                  'fast_disk': dj.config['custom'].get('suite2p_fast_tmp')[0],
                  'delete_bin': True,
                  'mesoscan': False,
                  'h5py': [],
                  'h5py_key': 'data',
                  'save_path0': [],
                  'subfolders': [],
                  'nplanes': 1,
                  'nchannels': 1,
                  'functional_chan': 1,
                  'tau': 1.0,
                  'fs': 15.3845,
                  'force_sktiff': False,
                  'preclassify': 0.0,
                  'save_mat': True,
                  'combined': True,
                  'aspect': 1.0,
                  'do_bidiphase': True,
                  'bidiphase': 0.0,
                  'do_registration': True,
                  'two_step_registration': True,
                  'norm_frames': True,
                  'keep_movie_raw': False,
                  'nimg_init': 1000,
                  'batch_size': 8000,
                  'maxregshift': 0.1,
                  'align_by_chan': 1,
                  'reg_tif': False,
                  'reg_tif_chan2': False,
                  'subpixel': 10,
                  'smooth_sigma_time': 1, 
                  'smooth_sigma': 1.15,
                  'th_badframes': 1.0,
                  'pad_fft': False,
                  'nonrigid': True,
                  'block_size': [64, 64],
                  'snr_thresh': 1.2,
                  'maxregshiftNR': 5.0,
                  '1Preg': False,
                  'spatial_hp': 50.0,
                  'pre_smooth': 2.0,
                  'spatial_taper': 50.0,
                  'roidetect': True,
                  'sparse_mode': True,
                  'denoise': True,
                  'diameter': 6,
                  'spatial_scale': 0,
                  'connected': True,
                  'nbinned': 9000,
                  'max_iterations': 30,
                  'threshold_scaling': 1.0,
                  'max_overlap': 0.75,
                  'high_pass': 100.0,
                  'inner_neuropil_radius': 2,
                  'min_neuropil_pixels': 350,
                  'allow_overlap': False,
                  'chan2_thres': 0.65,
                  'baseline': 'maximin',
                  'win_baseline': 60.0,
                  'sig_baseline': 10.0,
                  'prctile_baseline': 8.0,
                  'neucoeff': 0.7,
                  'xrange': np.array([0, 0]),
                  'yrange': np.array([0, 0])}


In [ ]:
imaging.ProcessingParamSet.insert_new_params(
    processing_method="suite2p",
    paramset_idx=0,
    params=params_suite2p,
    paramset_desc="TR: Mini2p (single channel, single plane, non-rigid, 15Hz)",
)


In [ ]:
# Insert the param_set - Trondheim Mini2p

# TODO: Parameter set needs to be updated with ScanInfo settings

params_suite2p = {'look_one_level_down': False,
                  'fast_disk': dj.config['custom'].get('suite2p_fast_tmp')[0],
                  'delete_bin': True,
                  'mesoscan': False,
                  'h5py': [],
                  'h5py_key': 'data',
                  'save_path0': [],
                  'subfolders': [],
                  'nplanes': 1,
                  'nchannels': 1,
                  'functional_chan': 1,
                  'tau': 1.0,
                  'fs': 15.3845,
                  'force_sktiff': False,
                  'preclassify': 0.0,
                  'save_mat': True,
                  'combined': True,
                  'aspect': 1.0,
                  'do_bidiphase': True,
                  'bidiphase': 0.0,
                  'do_registration': True,
                  'two_step_registration': True,
                  'norm_frames': True,
                  'keep_movie_raw': False,
                  'nimg_init': 1000,
                  'batch_size': 8000,
                  'maxregshift': 0.1,
                  'align_by_chan': 1,
                  'reg_tif': False,
                  'reg_tif_chan2': False,
                  'subpixel': 10,
                  'smooth_sigma_time': 1, 
                  'smooth_sigma': 1.15,
                  'th_badframes': 1.0,
                  'pad_fft': False,
                  'nonrigid': True,
                  'block_size': [64, 64],
                  'snr_thresh': 1.2,
                  'maxregshiftNR': 5.0,
                  '1Preg': False,
                  'spatial_hp': 50.0,
                  'pre_smooth': 2.0,
                  'spatial_taper': 50.0,
                  'roidetect': True,
                  'sparse_mode': True,
                  'denoise': True,
                  'diameter': 6,
                  'spatial_scale': 0,
                  'connected': True,
                  'nbinned': 9000,
                  'max_iterations': 30,
                  'threshold_scaling': 1.0,
                  'max_overlap': 0.75,
                  'high_pass': 100.0,
                  'inner_neuropil_radius': 2,
                  'min_neuropil_pixels': 350,
                  'allow_overlap': False,
                  'chan2_thres': 0.65,
                  'baseline': 'maximin',
                  'win_baseline': 60.0,
                  'sig_baseline': 10.0,
                  'prctile_baseline': 8.0,
                  'neucoeff': 0.7,
                  'xrange': np.array([0, 0]),
                  'yrange': np.array([0, 0])}


In [ ]:
imaging.ProcessingParamSet.insert_new_params(
    processing_method="suite2p",
    paramset_idx=0,
    params=params_suite2p,
    paramset_desc="TR: Mini2p (single channel, single plane, non-rigid, 15Hz)",
)


In [ ]:
# Insert the param_set - Bench2p

# TODO: Parameter set needs to be updated with ScanInfo settings

params_suite2p = {'look_one_level_down': False,
                  'fast_disk': dj.config['custom'].get('suite2p_fast_tmp')[0],
                  'delete_bin': True,
                  'mesoscan': False,
                  'h5py': [],
                  'h5py_key': 'data',
                  'save_path0': [],
                  'subfolders': [],
                  'nplanes': 1,
                  'nchannels': 1,
                  'functional_chan': 1,
                  'tau': 1.0,
                  'fs': 29.9784,
                  'force_sktiff': False,
                  'preclassify': 0.0,
                  'save_mat': True,
                  'combined': True,
                  'aspect': 1.0,
                  'do_bidiphase': True,
                  'bidiphase': 0.0,
                  'do_registration': True,
                  'two_step_registration': False,
                  'norm_frames': True,
                  'keep_movie_raw': False,
                  'nimg_init': 1000,
                  'batch_size': 8000,
                  'maxregshift': 0.1,
                  'align_by_chan': 1,
                  'reg_tif': True,
                  'reg_tif_chan2': False,
                  'subpixel': 10,
                  'smooth_sigma_time': 1, 
                  'smooth_sigma': 1.15,
                  'th_badframes': 1.0,
                  'pad_fft': False,
                  'nonrigid': True,
                  'block_size': [128, 128],
                  'snr_thresh': 1.2,
                  'maxregshiftNR': 5.0,
                  '1Preg': False,
                  'spatial_hp': 50.0,
                  'pre_smooth': 2.0,
                  'spatial_taper': 50.0,
                  'roidetect': True,
                  'sparse_mode': True,
                  'denoise': True,
                  'diameter': 12,
                  'spatial_scale': 0,
                  'connected': True,
                  'nbinned': 9000,
                  'max_iterations': 30,
                  'threshold_scaling': 1.0,
                  'max_overlap': 0.75,
                  'high_pass': 100.0,
                  'inner_neuropil_radius': 2,
                  'min_neuropil_pixels': 350,
                  'allow_overlap': False,
                  'chan2_thres': 0.65,
                  'baseline': 'maximin',
                  'win_baseline': 60.0,
                  'sig_baseline': 10.0,
                  'prctile_baseline': 8.0,
                  'neucoeff': 0.7,
                  'xrange': np.array([0, 0]),
                  'yrange': np.array([0, 0])}


In [ ]:
imaging.ProcessingParamSet.insert_new_params(
    processing_method="suite2p",
    paramset_idx=1,
    params=params_suite2p,
    paramset_desc="TR: Bench2p (single channel, single plane, non-rigid, 30Hz)",
)


In [ ]:
# Insert the param_set - RIGID Bench2p

# TODO: Parameter set needs to be updated with ScanInfo settings

params_suite2p = {'look_one_level_down': False,
                  'fast_disk': dj.config['custom'].get('suite2p_fast_tmp')[0],
                  'delete_bin': True,
                  'mesoscan': False,
                  'h5py': [],
                  'h5py_key': 'data',
                  'save_path0': [],
                  'subfolders': [],
                  'nplanes': 1,
                  'nchannels': 1,
                  'functional_chan': 1,
                  'tau': 1.0,
                  'fs': 29.9784,
                  'force_sktiff': False,
                  'preclassify': 0.0,
                  'save_mat': True,
                  'combined': True,
                  'aspect': 1.0,
                  'do_bidiphase': True,
                  'bidiphase': 0.0,
                  'do_registration': True,
                  'two_step_registration': False,
                  'norm_frames': True,
                  'keep_movie_raw': False,
                  'nimg_init': 1000,
                  'batch_size': 8000,
                  'maxregshift': 0.1,
                  'align_by_chan': 1,
                  'reg_tif': False,
                  'reg_tif_chan2': False,
                  'subpixel': 10,
                  'smooth_sigma_time': 1, 
                  'smooth_sigma': 1.15,
                  'th_badframes': 1.0,
                  'pad_fft': False,
                  'nonrigid': False,
                  'block_size': [128, 128],
                  'snr_thresh': 1.2,
                  'maxregshiftNR': 5.0,
                  '1Preg': False,
                  'spatial_hp': 50.0,
                  'pre_smooth': 2.0,
                  'spatial_taper': 50.0,
                  'roidetect': True,
                  'sparse_mode': True,
                  'denoise': True,
                  'diameter': 12,
                  'spatial_scale': 0,
                  'connected': True,
                  'nbinned': 9000,
                  'max_iterations': 30,
                  'threshold_scaling': 1.0,
                  'max_overlap': 0.75,
                  'high_pass': 100.0,
                  'inner_neuropil_radius': 2,
                  'min_neuropil_pixels': 350,
                  'allow_overlap': False,
                  'chan2_thres': 0.65,
                  'baseline': 'maximin',
                  'win_baseline': 60.0,
                  'sig_baseline': 10.0,
                  'prctile_baseline': 8.0,
                  'neucoeff': 0.7,
                  'xrange': np.array([0, 0]),
                  'yrange': np.array([0, 0])}


In [ ]:
imaging.ProcessingParamSet.insert_new_params(
    processing_method="suite2p",
    paramset_idx=1,
    params=params_suite2p,
    paramset_desc="TR: Bench2p (single channel, single plane, non-rigid, 30Hz)",
)


In [ ]:
imaging.ProcessingParamSet()

In [ ]:
imaging.ProcessingTask().delete()
imaging.ProcessingTask().drop()
imaging.ProcessingParamSet().drop()

### Create and Run a Processing Task

In [ ]:
scanquey = 'scan9FGLZLRI'

query =  scan.ScanInfo() & 'scan_id = "' + scanquey + '"'
sess_proc = query.fetch('session_id')[0]
scan_proc = query.fetch('scan_id')[0]

query2 = session.SessionDirectory() & 'session_id = "' + query.fetch('session_id')[0] + '"'
dir_proc = query2.fetch('session_dir')[0]

In [ ]:
scanquey = 'scan9FGLEFJ3'
b
sess_proc = query.fetch('session_id')[0]
scan_proc = query.fetch('scan_id')[0]

query2 = session.SessionDirectory() & 'session_id = "' + query.fetch('session_id')[0] + '"'
dir_proc = query2.fetch('session_dir')[0]

In [ ]:

query =  scan.ScanInfo() & 'scan_id = "' + scanquey + '"'
query

In [ ]:
scan.ScanInfo()

In [ ]:
imaging.ProcessingTask.insert1((sess_proc,
                                scan_proc,
                                0,
                                dir_proc,
                                'trigger'))

In [ ]:
imaging.ProcessingTask()

To run all unprocessed processing task we call populate on processing:

In [ ]:

scansi = "scan9FRDMV0W"
scan_key = (scan.Scan & f'scan_id = "{scansi}"').fetch('KEY')
# scans_to_process = (scan.ScanPath() * denoising.Denoising & scan_key).fetch('KEY')

In [ ]:
channels = (scan.ScanInfo & scan_key).fetch1('nchannels')
channels


In [ ]:
imaging.Processing & scan_key

In [ ]:
(imaging.Processing & scan_key & 'paramset_idx = 1060').delete()

In [ ]:
scan_key


In [ ]:
imaging.Processing.populate(display_progress=True, suppress_errors=True)

In [ ]:
for key in (imaging.ProcessingTask - imaging.Curation).fetch('KEY'):
    imaging.Curation().create1_from_processing_task(key)

In [ ]:
key = (imaging.ProcessingTask & scan_key & 'paramset_idx = 1060').fetch1('KEY')
imaging.Curation().create1_from_processing_task(key)

In [ ]:
imaging.Curation()

In [ ]:
populate_settings = {'display_progress': True, 'suppress_errors': True}

In [ ]:
imaging.MotionCorrection.populate(**populate_settings)

In [ ]:
imaging.Segmentation.populate(**populate_settings)

In [ ]:
imaging.MaskClassification.populate(**populate_settings)

In [ ]:
imaging.Fluorescence.populate(**populate_settings)

In [ ]:
imaging.Activity.populate(**populate_settings)

In [ ]:
imaging.ProcessingTask * imaging.Processing & session_key

In [ ]:
imaging.ProcessingTask()

In [ ]:
imaging.Processing()

In [ ]:
session_key = (session.Session & 'subject = "OPI-1681"').fetch('KEY')[0]

In [ ]:
scan.ScanInfo.ScanFile()


In [ ]:
scan.Scan & session_key

In [ ]:
scan.ScanInfo & session_key

In [ ]:
scan.ScanInfo.Field & session_key

In [ ]:
imaging.ProcessingParamSet()

In [ ]:
imaging.ProcessingTask * imaging.Processing & session_key

In [ ]:
imaging.Curation & session_key

Scanreader payload testing

In [ ]:
path = scan.ScanInfo.ScanFile().fetch('file_path')[0]
infoscan = scanreader.read_scan(path)

In [ ]:
infoscan.fpd

In [ ]:

infoscan.user_funtion


In [ ]:
scan.ScanInfo()

In [ ]:
session.Session()

In [ ]:
Sess